In [1]:
import sys
import os

# 모듈 자동 리로드 설정
%load_ext autoreload
%autoreload 2

# src 폴더 경로 설정
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

# 공통 함수 모듈 임포트
from src import common_utils as utils

utils.log("✅ 환경 설정 완료")

[00:25:07] ✅ [Config] 환경 설정 완료 (OS: Windows)
[00:25:07] ✅ 환경 설정 완료


In [2]:
# 1. 파일 경로 및 기간 설정
FILE_PATH = "../../data/10_processed/df_sha.parquet"

# 2. 데이터 로드
df = utils.load_parquet(FILE_PATH)


[00:25:30] 📂 로드 완료: 22,473,534 Rows (Selected 39 Columns)


In [3]:
# ==========================================================
# [검증 1] 전처리 전 원본 데이터(Raw) 확인
# ==========================================================
print("\n" + "="*60)
print("▶ [Step 1] 전처리 전 원본 데이터 구조 및 결측치 확인")
print("="*60)

# 1. 구조 확인 (info)
print("\n[1] 데이터 정보 (info)")
df.info()

# 2. 결측치 확인 (isna().sum())
print("\n[2] 컬럼별 결측치 개수 확인")
missing_raw = df.isna().sum()

# ID 컬럼은 통계에서 제외 (결과 Series에서만 drop)
cols_exclude = ['sha2_hash', 'SHA2_HASH', 'customer_id']
missing_raw = missing_raw.drop(labels=cols_exclude, errors='ignore')

# 결측치가 있는 컬럼만 출력
if missing_raw.sum() > 0:
    print("⚠️ 결측치가 발견되었습니다:")
    print(missing_raw[missing_raw > 0])
else:
    print("✅ 결측치 없음 (Raw Data)")
print("-" * 60)


▶ [Step 1] 전처리 전 원본 데이터 구조 및 결측치 확인

[1] 데이터 정보 (info)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22473534 entries, 0 to 22473533
Data columns (total 39 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   SHA2_HASH                  object 
 1   SVC_USE_DAYS_GRP           object 
 2   MEDIA_NM_GRP               object 
 3   PROD_NM_GRP                object 
 4   PROD_OLD_YN                object 
 5   PROD_ONE_PLUS_YN           object 
 6   AGMT_KIND_NM               object 
 7   STB_RES_1M_YN              object 
 8   SVOD_SCRB_CNT_GRP          object 
 9   PAID_CHNL_CNT_GRP          object 
 10  SCRB_PATH_NM_GRP           object 
 11  INHOME_RATE                object 
 12  AGMT_END_SEG               object 
 13  AGMT_END_YMD               object 
 14  TOTAL_USED_DAYS            int64  
 15  TV_SCRB                    float32
 16  ANALOG_SCRB                float32
 17  DIGITAL_SCRB               float32
 18  TOTAL_INTERNET_SCRB     

In [4]:
# 3. 전처리 (값 변환 -> 타입 변경 -> 이름 변경)
df = utils.data_preprocess(df)

# ==============================================================================
# [NEW] 3.5. 중복 제거 및 해지 이후 데이터 삭제 (Data Cleaning)
# ==============================================================================
utils.log("데이터 정제 시작: 중복 제거 및 해지 후 데이터 삭제...")

# (1) 단순 중복 제거 (동일한 ID, 동일한 월에 데이터가 여러 개면 하나만 남김)
# keep='last': 가장 최신 상태(보통 해지 시점)를 남기기 위함
df = df.sort_values(by=['SHA2_HASH', 'P_MT', 'derived_cancel_yn']) # 정렬
df = df.drop_duplicates(subset=['SHA2_HASH', 'P_MT'], keep='last')

# (2) "해지(1) 이후의 데이터" 삭제 로직
# 시나리오: 2월에 해지했는데 3, 4월 데이터가 또 있는 경우 -> 3, 4월 데이터는 삭제해야 함

# 해지한 이력이 있는 고객만 추출
churn_users = df[df['derived_cancel_yn'] == 1][['SHA2_HASH', 'P_MT']]
churn_users = churn_users.rename(columns={'P_MT': 'CHURN_MONTH'})

# 가장 처음 해지한 월(Min Churn Month)을 찾음 (재가입 이슈 방지)
first_churn = churn_users.groupby('SHA2_HASH')['CHURN_MONTH'].min().reset_index()

# 원본 데이터에 해지 월 정보를 붙임
df = df.merge(first_churn, on='SHA2_HASH', how='left')

# [삭제 조건] 해지 월(CHURN_MONTH)이 존재하면서, 현재 월(P_MT)이 해지 월보다 큰 경우 삭제
# 예: 해지월(202302) < 현재월(202303) -> 삭제 대상
mask_invalid = (df['CHURN_MONTH'].notnull()) & (df['P_MT'].astype(str) > df['CHURN_MONTH'].astype(str))
df_clean = df[~mask_invalid].copy()

# 불필요한 임시 컬럼 삭제
df_clean = df_clean.drop(columns=['CHURN_MONTH'])

utils.log(f"정제 전: {len(df):,}건 -> 정제 후: {len(df_clean):,}건 (삭제됨: {len(df)-len(df_clean):,}건)")
df = df_clean
# ==============================================================================

# 4. 결측치 제거
df = df.dropna()
utils.log(f"전처리 및 결측치 제거 완료: {len(df):,}건")


[00:26:07] 🚀 데이터 전처리(data_preprocess) 시작...
[00:26:52] ✨ 전처리 완료: 37개 컬럼 변환 및 정리됨
[00:26:59] 데이터 정제 시작: 중복 제거 및 해지 후 데이터 삭제...
[00:27:48] 정제 전: 22,473,534건 -> 정제 후: 21,354,568건 (삭제됨: 1,118,966건)
[00:27:59] 전처리 및 결측치 제거 완료: 18,500,852건


In [5]:
# ==========================================================
# [검증 2] 전처리 및 정제 후 데이터(Clean) 확인
# ==========================================================
print("\n" + "="*60)
print("▶ [Step 2] 전처리 완료 후 최종 데이터 구조 및 결측치 확인")
print("="*60)

# 1. 구조 확인 (info)
print("\n[1] 데이터 정보 (info)")
df.info()

# 2. 결측치 확인 (isna().sum())
print("\n[2] 컬럼별 결측치 개수 확인")
missing_clean = df.isna().sum()

# ID 컬럼은 통계에서 제외 (결과 Series에서만 drop)
cols_exclude = ['sha2_hash', 'SHA2_HASH', 'customer_id']
missing_clean = missing_clean.drop(labels=cols_exclude, errors='ignore')

# 결측치가 있는 컬럼만 출력
if missing_clean.sum() > 0:
    print("⚠️ 아직 결측치가 남아있습니다:")
    print(missing_clean[missing_clean > 0])
else:
    print("✅ 모든 데이터가 깨끗합니다! (결측치 0개)")
print("-" * 60)


▶ [Step 2] 전처리 완료 후 최종 데이터 구조 및 결측치 확인

[1] 데이터 정보 (info)
<class 'pandas.core.frame.DataFrame'>
Index: 18500852 entries, 0 to 22473533
Data columns (total 39 columns):
 #   Column                             Dtype  
---  ------                             -----  
 0   SHA2_HASH                          object 
 1   DERIVED_SVC_USE_DAYS_GRP           object 
 2   DERIVED_MEDIA_NM_GRP               object 
 3   DERIVED_PROD_NM_GRP                object 
 4   DERIVED_PROD_OLD_YN                object 
 5   DERIVED_PROD_ONE_PLUS_YN           object 
 6   DERIVED_AGMT_KIND_NM               object 
 7   DERIVED_STB_RES_1M_YN              object 
 8   DERIVED_SVOD_SCRB_CNT_GRP          float64
 9   DERIVED_PAID_CHNL_CNT_GRP          float64
 10  DERIVED_SCRB_PATH_NM_GRP           object 
 11  DERIVED_INHOME_RATE                float64
 12  DERIVED_AGMT_END_SEG               object 
 13  DERIVED_AGMT_END_YMD               object 
 14  DERIVED_TOTAL_USED_DAYS            float64
 15  DERIVED_TV

In [11]:
# 전처리 완료된 데이터 저장
SAVE_PATH = "../../data/10_processed/XGBoost_dataset.parquet"
df.to_parquet(SAVE_PATH, index=False)

utils.log(f"💾 전처리된 데이터 저장 완료: {SAVE_PATH}")

[23:46:15] 💾 전처리된 데이터 저장 완료: ../../data/10_processed/XGBoost_dataset.parquet
